## CELL 00 — Notebook purpose and pipeline

This notebook implements the **S³ Spectral-Spatial Domain-Adaptive EEG classification pipeline** from the supplied implementation.

### What the pipeline does
The model takes a 4-second, 22-channel EEG trial and performs:

**Raw EEG → per-trial channel Z-score → learnable Sinc filter bank → dynamic graph neural network → bidirectional GRU temporal modeling → squeeze-and-excitation attention → 4-class classifier**

At the same time, the learned representation is sent to:
- a **domain/subject classifier through a Gradient Reversal Layer (GRL)** for subject/domain alignment;
- a **supervised contrastive projection head** to pull same-class embeddings together.

After supervised source-domain training, the held-out target subject is adapted using **AdaBN-style target-statistics updating** with unlabeled target trials.

### Important implementation caveats
The supplied class is called `SimplifiedBiMamba`, but the actual sequence block is a **bidirectional GRU**, not a true Mamba/state-space model. The supplied notebook also does **not** implement the latent diffusion module mentioned in its manuscript description. These are implementation facts from the source notebook and should be stated accurately in a paper. fileciteturn1file0L19-L22

## CELL 01 — Imports and environment

This cell imports the numerical, EEG, deep-learning, and evaluation libraries.

The important packages are:
- **MNE** for reading PhysioNet EEGMMIDB EDF files, annotations, and epoching.
- **PyTorch** for the model and training.
- **scikit-learn** for accuracy, Cohen's kappa, classification metrics, ROC/AUC, and t-SNE.
- **pandas / NumPy / Matplotlib** for experiment logging and figures.

In [1]:
# ============================================================
# CELL 01 — Imports and environment
# ============================================================

# ============================================================
# S3 Spectral-Spatial Domain Adaptation EEG Research Pipeline
# Code-aligned with the supplied implementation
#
# IMPORTANT IMPLEMENTATION NOTE
# ------------------------------------------------------------
# The supplied model contains:
#   1) per-trial channel-wise Z-score normalization
#   2) learnable Sinc filter bank
#   3) dynamic graph neural network (DGNN)
#   4) "SimplifiedBiMamba" class implemented with a bidirectional GRU
#   5) squeeze-and-excitation attention
#   6) class head + GRL domain head + supervised contrastive head
#   7) AdaBN test-time adaptation
#
# It does NOT implement the latent diffusion module described in the
# attached reference manuscript, and its "Mamba" block is currently
# a bidirectional GRU approximation. The paper should therefore not
# claim true Mamba or diffusion results unless those modules are
# implemented separately.
# ============================================================

from pathlib import Path
import os, math, json, random, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mne
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, confusion_matrix,
    classification_report, roc_curve, auc
)
from sklearn.manifold import TSNE

warnings.filterwarnings("ignore")
mne.set_log_level("ERROR")


## CELL 02 — Configuration and reproducibility

This cell defines the experimental settings.

The source configuration uses:
- 109 possible subjects
- runs 4, 6, 8, 10, 12, and 14
- 250 Hz sampling
- 4-second trials
- 22 EEG channels
- 4 classes
- batch size 64
- 100 training epochs
- AdamW with learning rate `1e-3`
- cosine annealing
- label smoothing, domain loss, and supervised contrastive loss

A fixed random seed of 42 is used. The code automatically selects CUDA, then Apple MPS, then CPU. fileciteturn1file0L76-L137

In [2]:
# ============================================================
# CELL 02 — Configuration and reproducibility
# ============================================================

# -----------------------------
# 0. CONFIGURATION
# -----------------------------
SEED = 42
DATA_DIR = "./eegmmidb"

TOTAL_SUBJECTS = 109
NUM_TEST_FOLDS = 1
NUM_TRAIN_SUBJECTS = 99

RUNS = [4, 6, 8, 10, 12, 14]
TMIN = 0.0
TMAX = 4.0
FS = 250.0
N_CHANNELS = 22
N_CLASSES = 4

BATCH_SIZE = 64
NUM_EPOCHS = 2
LR = 1e-3
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
DOMAIN_WEIGHT = 1.0
SUPCON_WEIGHT = 0.5
SUPCON_TEMP = 0.07
GRAD_CLIP = 1.0

NUM_FILTERS = 10
SINC_KERNEL = 81
SPATIAL_DIM = 64
DOMAIN_CLASSES = TOTAL_SUBJECTS

RESULTS_DIR = Path("./results_s3_da")
FIG_DIR = RESULTS_DIR / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = [
    "Left Fist",
    "Right Fist",
    "Both Fists",
    "Both Feet",
]

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("Device:", DEVICE)
print("Torch:", torch.__version__)


Device: mps
Torch: 2.10.0


## CELL 03 — Dataset loader

`EEGMMIDB_Dataset` scans subject folders (`S001`, `S002`, …), loads the selected EDF runs, keeps the first 22 channels, resamples to 250 Hz, reads annotation events, and creates 4-second epochs.

The run-dependent label mapping is:

| Runs | T1 | T2 |
|---|---|---|
| 4, 8, 12 | Left Fist = 0 | Right Fist = 1 |
| 6, 10, 14 | Both Fists = 2 | Both Feet = 3 |

The dataset also stores a **subject ID** for the domain-adaptation loss and the run ID for traceability. Each trial is then normalized independently, channel by channel, using its own temporal mean and standard deviation. fileciteturn1file0L143-L240

In [3]:
# ============================================================
# CELL 03 — FIXED EEGMMIDB DATASET LOADER
# ============================================================
#
# IMPORTANT FIX:
# MNE's events_from_annotations() returns the ACTUAL numeric
# event codes associated with annotation descriptions such as
# "T1" and "T2".
#
# We must NOT manually use:
#     {"T1": 0, "T2": 1}
#
# because 0/1 may not be the actual event IDs in the EDF file.
#
# This version:
#   1. Reads the EDF
#   2. Keeps the first 22 EEG channels
#   3. Resamples to 250 Hz
#   4. Reads annotation/event IDs
#   5. Finds the actual event code for T1/T2
#   6. Converts them into our own class labels:
#
#      Runs 4,8,12:
#          T1 -> 0 = Left Fist
#          T2 -> 1 = Right Fist
#
#      Runs 6,10,14:
#          T1 -> 2 = Both Fists
#          T2 -> 3 = Both Feet
#
#   7. Epochs each trial from 0 to 4 seconds
#   8. Performs per-trial, per-channel Z-score normalization
#
# ============================================================

class EEGMMIDB_Dataset(Dataset):

    def __init__(
        self,
        data_dir,
        subjects,
        runs=RUNS,
        tmin=TMIN,
        tmax=TMAX
    ):
        self.data_dir = str(data_dir)
        self.subjects = list(subjects)
        self.runs = list(runs)
        self.tmin = tmin
        self.tmax = tmax

        self.epochs = []
        self.labels = []
        self.subject_ids = []
        self.run_ids = []

        self.load_data()

    # --------------------------------------------------------
    # Find actual MNE event code for annotation description
    # --------------------------------------------------------
    @staticmethod
    def _find_event_code(event_id_dict, target_name):
        """
        Find the numeric event code corresponding to an
        annotation such as T1 or T2.

        MNE may return keys with small formatting differences,
        so matching is made robust.
        """

        target_name = str(target_name).strip().upper()

        for description, code in event_id_dict.items():

            desc = str(description).strip().upper()

            # Exact match
            if desc == target_name:
                return int(code)

            # Handle possible annotation variants such as:
            # "T1", "T1 ", "T1/..."
            if desc.startswith(target_name):
                remainder = desc[len(target_name):]

                if (
                    remainder == ""
                    or remainder.startswith("/")
                    or remainder.startswith("-")
                    or remainder.startswith("_")
                    or remainder.isspace()
                ):
                    return int(code)

        return None

    # --------------------------------------------------------
    # Load all subjects/runs
    # --------------------------------------------------------
    def load_data(self):

        total_loaded = 0

        for sub in self.subjects:

            sub_folder = f"S{sub:03d}"
            sub_path = os.path.join(
                self.data_dir,
                sub_folder
            )

            if not os.path.isdir(sub_path):
                print(
                    f"[WARN] Subject folder not found: "
                    f"{sub_path}"
                )
                continue

            for run in self.runs:

                edf_file = os.path.join(
                    sub_path,
                    f"{sub_folder}R{run:02d}.edf"
                )

                if not os.path.isfile(edf_file):
                    print(
                        f"[WARN] Missing EDF: "
                        f"{edf_file}"
                    )
                    continue

                try:

                    # ------------------------------------------------
                    # 1. Load EDF
                    # ------------------------------------------------
                    raw = mne.io.read_raw_edf(
                        edf_file,
                        preload=True,
                        verbose=False
                    )

                    # ------------------------------------------------
                    # 2. Keep first 22 channels
                    # ------------------------------------------------
                    if len(raw.ch_names) < N_CHANNELS:
                        print(
                            f"[WARN] {sub_folder} R{run:02d}: "
                            f"only {len(raw.ch_names)} channels found; "
                            f"expected {N_CHANNELS}."
                        )
                        continue

                    raw.pick(
                        raw.ch_names[:N_CHANNELS]
                    )

                    # ------------------------------------------------
                    # 3. Resample to 250 Hz
                    # ------------------------------------------------
                    raw.resample(
                        FS,
                        npad="auto"
                    )

                    # ------------------------------------------------
                    # 4. Extract annotations/events
                    # ------------------------------------------------
                    events, event_id_dict = (
                        mne.events_from_annotations(
                            raw,
                            verbose=False
                        )
                    )

                    # ------------------------------------------------
                    # 5. Determine which labels this run represents
                    # ------------------------------------------------
                    if run in [4, 8, 12]:

                        # Left/right fist imagery
                        target_class_t1 = 0
                        target_class_t2 = 1

                    elif run in [6, 10, 14]:

                        # Both fists / both feet imagery
                        target_class_t1 = 2
                        target_class_t2 = 3

                    else:
                        continue

                    # ------------------------------------------------
                    # 6. Find ACTUAL numeric MNE event IDs
                    # ------------------------------------------------
                    t1_code = self._find_event_code(
                        event_id_dict,
                        "T1"
                    )

                    t2_code = self._find_event_code(
                        event_id_dict,
                        "T2"
                    )

                    # ------------------------------------------------
                    # 7. Validate that T1/T2 exist
                    # ------------------------------------------------
                    if t1_code is None or t2_code is None:

                        available_events = list(
                            event_id_dict.keys()
                        )

                        print(
                            f"[WARN] {sub_folder} R{run:02d}: "
                            f"T1/T2 not found. "
                            f"Available annotations: "
                            f"{available_events}"
                        )

                        continue

                    # ------------------------------------------------
                    # 8. Create MNE event selection using ACTUAL codes
                    #
                    # IMPORTANT:
                    # The values here are the event IDs returned by MNE,
                    # NOT our machine-learning class labels.
                    # ------------------------------------------------
                    event_selection = {
                        "T1": t1_code,
                        "T2": t2_code
                    }

                    # ------------------------------------------------
                    # 9. Epoch from 0 → 4 seconds
                    # ------------------------------------------------
                    epochs_mne = mne.Epochs(
                        raw,
                        events,
                        event_id=event_selection,
                        tmin=self.tmin,
                        tmax=self.tmax - 1.0 / FS,
                        baseline=None,
                        preload=True,
                        reject_by_annotation=True,
                        verbose=False
                    )

                    if len(epochs_mne) == 0:
                        print(
                            f"[WARN] {sub_folder} R{run:02d}: "
                            f"T1/T2 were found but produced "
                            f"zero epochs."
                        )
                        continue

                    # ------------------------------------------------
                    # 10. Extract EEG data
                    # ------------------------------------------------
                    data = epochs_mne.get_data()

                    # ------------------------------------------------
                    # 11. Map ACTUAL event codes to our class labels
                    #
                    # epochs_mne.events[:, -1] contains the ACTUAL
                    # numeric MNE event code.
                    # We convert those codes to our 0..3 classes.
                    # ------------------------------------------------
                    actual_codes = epochs_mne.events[:, -1]

                    for trial_idx in range(len(data)):

                        actual_code = int(
                            actual_codes[trial_idx]
                        )

                        # T1
                        if actual_code == t1_code:

                            class_label = target_class_t1

                        # T2
                        elif actual_code == t2_code:

                            class_label = target_class_t2

                        else:
                            # Should never happen because Epochs
                            # selected only T1/T2.
                            continue

                        # ------------------------------------------------
                        # 12. Store trial
                        # ------------------------------------------------
                        self.epochs.append(
                            data[trial_idx].astype(
                                np.float32
                            )
                        )

                        self.labels.append(
                            int(class_label)
                        )

                        # Subject domain label:
                        # S001 -> 0
                        # S002 -> 1
                        # ...
                        # S109 -> 108
                        self.subject_ids.append(
                            int(sub - 1)
                        )

                        self.run_ids.append(
                            int(run)
                        )

                        total_loaded += 1

                    # ------------------------------------------------
                    # Optional per-run diagnostic
                    # ------------------------------------------------
                    print(
                        f"[OK] {sub_folder} R{run:02d} | "
                        f"T1={t1_code} -> class {target_class_t1} | "
                        f"T2={t2_code} -> class {target_class_t2} | "
                        f"epochs={len(epochs_mne)}"
                    )

                except Exception as e:

                    print(
                        f"[WARN] {sub_folder} R{run:02d}: "
                        f"{type(e).__name__}: {e}"
                    )

        print(
            f"\nDataset loading complete: "
            f"{total_loaded} trials"
        )

    # --------------------------------------------------------
    # Dataset length
    # --------------------------------------------------------
    def __len__(self):
        return len(self.epochs)

    # --------------------------------------------------------
    # Dataset item
    # --------------------------------------------------------
    def __getitem__(self, idx):

        x = torch.tensor(
            self.epochs[idx],
            dtype=torch.float32
        )

        y = torch.tensor(
            self.labels[idx],
            dtype=torch.long
        )

        s = torch.tensor(
            self.subject_ids[idx],
            dtype=torch.long
        )

        # ----------------------------------------------------
        # Per-trial, per-channel Z-score normalization
        #
        # x shape:
        #     [22, 1000]
        #
        # mean/std computed independently for each channel
        # ----------------------------------------------------
        mean = x.mean(
            dim=1,
            keepdim=True
        )

        std = x.std(
            dim=1,
            keepdim=True
        )

        x = (
            x - mean
        ) / (
            std + 1e-6
        )

        return x, y, s


# ============================================================
# SUBJECT DISCOVERY
# ============================================================

def discover_available_subjects(
    data_dir,
    max_subjects=TOTAL_SUBJECTS
):

    available = []

    for s in range(
        1,
        max_subjects + 1
    ):

        subject_dir = (
            Path(data_dir) /
            f"S{s:03d}"
        )

        if subject_dir.is_dir():

            available.append(s)

    return available


# ============================================================
# DATASET DIAGNOSTIC TEST
# ============================================================

print("=" * 80)
print("DATASET LOADER TEST")
print("=" * 80)

available_subjects = discover_available_subjects(
    DATA_DIR,
    TOTAL_SUBJECTS
)

print(
    f"Available subject folders: "
    f"{len(available_subjects)}"
)

if len(available_subjects) > 0:

    # Load a single subject first.
    # This allows us to verify the event mapping
    # before launching the full 10-fold experiment.

    test_subject = available_subjects[0]

    print(
        f"\nTesting subject: "
        f"S{test_subject:03d}"
    )

    test_dataset = EEGMMIDB_Dataset(
        DATA_DIR,
        subjects=[test_subject]
    )

    print("\nDataset test results:")
    print(
        "  Number of trials:",
        len(test_dataset)
    )

    if len(test_dataset) > 0:

        x_test, y_test, s_test = (
            test_dataset[0]
        )

        print(
            "  Trial shape:",
            tuple(x_test.shape)
        )

        print(
            "  Class label:",
            int(y_test)
        )

        print(
            "  Subject domain:",
            int(s_test)
        )

        print(
            "  Expected shape:",
            (N_CHANNELS, int(FS * TMAX))
        )

        print(
            "\n[OK] Dataset loader is working."
        )

    else:

        print(
            "\n[ERROR] No trials were loaded "
            "for the test subject."
        )

else:

    print(
        "\n[ERROR] No subject folders found."
    )

DATASET LOADER TEST
Available subject folders: 109

Testing subject: S001
[OK] S001 R04 | T1=2 -> class 0 | T2=3 -> class 1 | epochs=15
[OK] S001 R06 | T1=2 -> class 2 | T2=3 -> class 3 | epochs=15
[OK] S001 R08 | T1=2 -> class 0 | T2=3 -> class 1 | epochs=15
[OK] S001 R10 | T1=2 -> class 2 | T2=3 -> class 3 | epochs=15
[OK] S001 R12 | T1=2 -> class 0 | T2=3 -> class 1 | epochs=15
[OK] S001 R14 | T1=2 -> class 2 | T2=3 -> class 3 | epochs=15

Dataset loading complete: 90 trials

Dataset test results:
  Number of trials: 90
  Trial shape: (22, 1000)
  Class label: 1
  Subject domain: 0
  Expected shape: (22, 1000)

[OK] Dataset loader is working.


## CELL 04 — Gradient Reversal and learnable Sinc filter bank

This part starts the feature extractor.

### Gradient Reversal Layer
During forward propagation the GRL behaves like an identity operation. During backpropagation it multiplies the gradient by `-lambda`. Therefore:
- the domain classifier learns to predict the subject;
- the feature extractor receives the **opposite** gradient and is encouraged to make subjects harder to distinguish.

### Learnable Sinc filter bank
The model learns two frequency cutoffs for each of 10 filters. Each filter is constructed as a band-pass Sinc kernel and applied independently to every EEG channel.

Input shape:
`(B, 22, 1000)`

Output shape:
`(B, 10, 22, 1000)`

So the raw temporal signal becomes a learned set of 10 spectral representations. fileciteturn1file0L254-L309

In [4]:
# ============================================================
# CELL 04 — GRL and SincFilterBank
# ============================================================

# -----------------------------
# 2. ARCHITECTURE
# -----------------------------

class GradientReversalLayer(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_grl):
        ctx.lambda_grl = lambda_grl
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.lambda_grl, None

def grl(x, lambda_grl=1.0):
    return GradientReversalLayer.apply(x, lambda_grl)





## CELL 05 — Dynamic graph neural network

`DGNN` builds an adaptive channel graph from the spectral features.

For each trial it:
1. averages over time to obtain channel descriptors;
2. projects the descriptors into query and key spaces;
3. computes pairwise channel similarity;
4. applies softmax to obtain a dynamic adjacency matrix;
5. adds self-connections;
6. degree-normalizes the adjacency matrix;
7. mixes information across EEG channels;
8. projects the resulting 22-node representation into a 64-dimensional spatial feature space.

The returned tensor has shape:
`(B, 10 bands, 64 spatial features, T)`. fileciteturn1file0L312-L343

In [5]:
# ============================================================
# CELL 05 — Dynamic GNN
# ============================================================

class SincFilterBank(nn.Module):
    def __init__(self, in_channels=22, num_filters=10, kernel_size=81, sample_rate=250):
        super().__init__()
        self.num_filters = num_filters
        self.kernel_size = kernel_size
        self.sample_rate = sample_rate

        # Same initialization family as the supplied code.
        self.f1 = nn.Parameter(torch.rand(num_filters) * 10 + 5)
        self.f2 = nn.Parameter(torch.rand(num_filters) * 20 + 15)

    def forward(self, x):
        B, C, T = x.shape

        n = torch.arange(
            -(self.kernel_size // 2),
            (self.kernel_size // 2) + 1,
            device=x.device,
            dtype=x.dtype
        )

        filters = []
        for i in range(self.num_filters):
            # Sorted positive cutoffs make the implementation numerically safer
            # without changing the conceptual learnable-Sinc design.
            lo = torch.minimum(self.f1[i], self.f2[i] - 1e-3).clamp(0.5, 70.0)
            hi = torch.maximum(self.f2[i], self.f1[i] + 1e-3).clamp(1.0, 95.0)

            f1_scaled = lo / self.sample_rate
            f2_scaled = hi / self.sample_rate

            w = (
                2 * f2_scaled * torch.sinc(2 * f2_scaled * n)
                - 2 * f1_scaled * torch.sinc(2 * f1_scaled * n)
            )
            filters.append(w.unsqueeze(0).unsqueeze(0))

        filters = torch.cat(filters, dim=0)
        x_reshaped = x.reshape(B * C, 1, T)
        out = F.conv1d(x_reshaped, filters, padding="same")

        return out.reshape(B, C, self.num_filters, T).permute(0, 2, 1, 3)





## CELL 06 — Temporal block and SE attention

The class named `SimplifiedBiMamba` is a bidirectional GRU.

After averaging the 10 spectral bands, the tensor is:
`(B, 64, T)`.

The GRU reads the time axis in both directions and returns another `(B, 64, T)` representation.

`SEAttention` then:
1. averages each feature over time;
2. uses a small bottleneck MLP and sigmoid to generate feature weights;
3. reweights the temporal features;
4. averages over time to produce the final 64-dimensional embedding.

This 64-D representation is the shared feature used by the task, domain, and contrastive objectives. fileciteturn1file0L346-L381

In [10]:
# ============================================================
# CELL 06 — TEMPORAL MODEL + SE ATTENTION
# ============================================================

class SimplifiedBiMamba(nn.Module):
    """
    IMPORTANT:
    Despite the name, this is NOT a true Mamba/SSM implementation.

    It uses a bidirectional GRU:
        input  : [B, D, T]
        output : [B, D, T]

    With D=64:
        GRU hidden size = 32 per direction
        bidirectional = 64 output features
    """

    def __init__(self, d_model=64):
        super().__init__()

        self.ssm = nn.GRU(
            input_size=d_model,
            hidden_size=d_model // 2,
            batch_first=True,
            bidirectional=True
        )

    def forward(self, x):
        """
        Input:
            x = [B, D, T]

        GRU expects:
            [B, T, D]

        Output:
            [B, D, T]
        """

        # [B, D, T] -> [B, T, D]
        x_seq = x.transpose(1, 2)

        # Bidirectional GRU
        out, _ = self.ssm(x_seq)

        # [B, T, D] -> [B, D, T]
        return out.transpose(1, 2)


# ============================================================
# SQUEEZE-AND-EXCITATION ATTENTION
# ============================================================

class SEAttention(nn.Module):
    """
    Squeeze-and-Excitation attention.

    Input:
        [B, C, T]

    Steps:
        1. Global average pooling over time
        2. Bottleneck fully-connected layer
        3. ReLU
        4. Expansion layer
        5. Sigmoid channel weights
        6. Re-weight original feature map
        7. Average over time

    Output:
        [B, C]
    """

    def __init__(self, channel=64, reduction=16):
        super().__init__()

        # Make sure the bottleneck does not become zero
        hidden_dim = max(1, channel // reduction)

        self.fc = nn.Sequential(
            nn.Linear(
                channel,
                hidden_dim,
                bias=False
            ),

            nn.ReLU(inplace=True),

            nn.Linear(
                hidden_dim,
                channel,
                bias=False
            ),

            nn.Sigmoid()
        )

    def forward(self, x):

        # ----------------------------------------------------
        # x shape:
        # [B, C, T]
        # ----------------------------------------------------
        b, c, t = x.size()

        # ----------------------------------------------------
        # Squeeze:
        # Global average pooling across time
        #
        # [B, C, T]
        #      ↓
        # [B, C]
        # ----------------------------------------------------
        y = x.mean(dim=2)

        # ----------------------------------------------------
        # Excitation:
        # Learn one importance weight per feature channel
        #
        # [B, C]
        #      ↓
        # [B, C]
        # ----------------------------------------------------
        s = self.fc(y)

        # ----------------------------------------------------
        # Reshape:
        # [B, C] -> [B, C, 1]
        # ----------------------------------------------------
        s = s.view(b, c, 1)

        # ----------------------------------------------------
        # Re-weight temporal features
        #
        # [B, C, T] × [B, C, 1]
        # ----------------------------------------------------
        weighted_x = x * s

        # ----------------------------------------------------
        # Final temporal pooling
        #
        # [B, C, T]
        #      ↓
        # [B, C]
        # ----------------------------------------------------
        return weighted_x.mean(dim=2)


# ============================================================
# VERIFY CELL 06
# ============================================================

print("=" * 70)
print("CELL 06 CHECK")
print("=" * 70)

print("SimplifiedBiMamba defined :", "SimplifiedBiMamba" in globals())
print("SEAttention defined       :", "SEAttention" in globals())

# Small tensor test
_test_x = torch.randn(
    2,
    SPATIAL_DIM,
    int(FS * TMAX)
).to(DEVICE)

with torch.no_grad():

    _temporal_model = SimplifiedBiMamba(
        d_model=SPATIAL_DIM
    ).to(DEVICE)

    _se_model = SEAttention(
        channel=SPATIAL_DIM
    ).to(DEVICE)

    _temporal_out = _temporal_model(_test_x)
    _se_out = _se_model(_temporal_out)

print("Input shape               :", tuple(_test_x.shape))
print("BiGRU output shape        :", tuple(_temporal_out.shape))
print("SE output shape           :", tuple(_se_out.shape))

del _test_x
del _temporal_model
del _se_model
del _temporal_out
del _se_out

print("\n[OK] Cell 06 completed successfully.")

CELL 06 CHECK
SimplifiedBiMamba defined : True
SEAttention defined       : True
Input shape               : (2, 64, 1000)
BiGRU output shape        : (2, 64, 1000)
SE output shape           : (2, 64)

[OK] Cell 06 completed successfully.


## CELL 07 — Full S³MambaDA model

`S3MambaDA` connects the complete feature extractor and the three training heads:

**Input → Sinc → DGNN → band pooling → BiGRU → SE attention → 64-D embedding**

Then:
- **Classifier:** 64 → 4 class logits.
- **Domain classifier:** GRL → 64 → 32 → 109 subject logits.
- **SupCon projection:** 64 → 128 → 128, followed by L2 normalization.

The forward pass therefore returns three objects:
`class_logits, domain_logits, z_proj`. fileciteturn1file0L384-L436

In [11]:
# ============================================================
# CELL 07 — Full S³MambaDA model
# ============================================================

class S3MambaDA(nn.Module):
    """
    Full model from the supplied implementation.

    NOTE:
    `SimplifiedBiMamba` is implemented with a bidirectional GRU,
    not a true Mamba state-space model.
    """

    def __init__(self, num_classes=4, num_subjects=109):
        super().__init__()

        self.sinc_filter = SincFilterBank(
            in_channels=N_CHANNELS,
            num_filters=NUM_FILTERS,
            kernel_size=SINC_KERNEL,
            sample_rate=int(FS)
        )

        self.dgnn = DGNN(
            num_filters=NUM_FILTERS,
            in_nodes=N_CHANNELS,
            out_nodes=SPATIAL_DIM
        )

        self.mamba = SimplifiedBiMamba(
            d_model=SPATIAL_DIM
        )

        self.se_attention = SEAttention(
            channel=SPATIAL_DIM
        )

        # EEG class prediction
        self.classifier = nn.Sequential(
            nn.BatchNorm1d(SPATIAL_DIM),
            nn.Linear(SPATIAL_DIM, num_classes)
        )

        # Subject/domain prediction
        self.domain_classifier = nn.Sequential(
            nn.Linear(SPATIAL_DIM, 32),
            nn.ReLU(),
            nn.Linear(32, num_subjects)
        )

        # Supervised contrastive projection
        self.supcon_proj = nn.Sequential(
            nn.Linear(SPATIAL_DIM, 128),
            nn.ReLU(),
            nn.Linear(128, 128)
        )

    def forward(self, x, lambda_grl=1.0):

        # ----------------------------------------------------
        # Input
        # x = [Batch, Channels, Time]
        # Example = [B, 22, 1000]
        # ----------------------------------------------------

        # Spectral decomposition
        f_out = self.sinc_filter(x)
        # [B, 10, 22, T]

        # Spatial graph learning
        s_out = self.dgnn(f_out)
        # [B, 10, 64, T]

        # Average learned spectral bands
        pool_out = s_out.mean(dim=1)
        # [B, 64, T]

        # Temporal modeling
        t_out = self.mamba(pool_out)
        # [B, 64, T]

        # SE channel attention + temporal pooling
        z = self.se_attention(t_out)
        # [B, 64]

        # ----------------------------------------------------
        # Classification branch
        # ----------------------------------------------------
        class_logits = self.classifier(z)

        # ----------------------------------------------------
        # Domain-adversarial branch
        # ----------------------------------------------------
        z_grl = grl(z, lambda_grl)
        domain_logits = self.domain_classifier(z_grl)

        # ----------------------------------------------------
        # Supervised contrastive branch
        # ----------------------------------------------------
        z_proj = F.normalize(
            self.supcon_proj(z),
            p=2,
            dim=1
        )

        return class_logits, domain_logits, z_proj

## CELL 08 — Losses and AdaBN adaptation

The training objective is the sum of three components:

`Total = Classification Loss + 1.0 × Domain Loss + 0.5 × SupCon Loss`

The classification loss uses cross-entropy with label smoothing. The domain loss predicts the subject identity. The supervised contrastive loss uses class labels so trials from the same class act as positives.

After training on source subjects, `apply_adabn()` collects unlabeled target trials and refreshes BatchNorm running statistics from target data. This is a **test-time statistics adaptation** step rather than supervised target-label fine-tuning. fileciteturn1file0L442-L512

In [12]:
# ============================================================
# CELL 08 — SupCon loss and AdaBN adaptation
# ============================================================

# -----------------------------
# 3. LOSSES
# -----------------------------
class SupConLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        device = features.device
        batch_size = features.shape[0]

        sim = torch.matmul(features, features.T) / self.temperature

        labels = labels.contiguous().view(-1, 1)
        mask = torch.eq(labels, labels.T).float().to(device)

        logits_mask = torch.ones_like(mask)
        logits_mask.fill_diagonal_(0)
        mask = mask * logits_mask

        exp_logits = torch.exp(sim) * logits_mask
        log_prob = sim - torch.log(exp_logits.sum(1, keepdim=True) + 1e-6)

        positives = mask.sum(1)
        mean_log_prob_pos = (mask * log_prob).sum(1) / (positives + 1e-6)

        valid = positives > 0
        if valid.any():
            return -mean_log_prob_pos[valid].mean()
        return torch.zeros((), device=device, requires_grad=True)


def apply_adabn(model, target_dataloader, device, adaptation_trials=20):
    model.eval()

    target_samples = []
    trials_count = 0

    for x, _, _ in target_dataloader:
        target_samples.append(x)
        trials_count += x.size(0)
        if trials_count >= adaptation_trials:
            break

    if not target_samples:
        return model

    target_x = torch.cat(target_samples, dim=0)[:adaptation_trials].to(device)

    bn_modules = [
        m for m in model.modules()
        if isinstance(m, nn.modules.batchnorm._BatchNorm)
    ]

    if not bn_modules:
        return model

    saved = []
    for module in bn_modules:
        saved.append((module.training, module.momentum))
        module.reset_running_stats()
        module.momentum = 1.0
        module.train()

    with torch.no_grad():
        _ = model(target_x, lambda_grl=0.0)

    for module, (was_training, old_momentum) in zip(bn_modules, saved):
        module.momentum = 0.1
        module.eval()

    model.eval()
    return model


## CELL 09 — Model smoke test

Before launching the expensive experiment, this cell creates a random tensor with the expected EEG size and verifies that the complete model executes.

Expected:
- input `(2, 22, 1000)`
- 4 class logits
- 109 subject/domain logits
- 128-dimensional contrastive projection

It also reports the parameter count. fileciteturn1file0L515-L532

In [13]:
# ============================================================
# CELL 09 — Smoke test
# ============================================================

# -----------------------------
# 4. MODEL SMOKE TEST
# -----------------------------
def smoke_test():
    model = S3MambaDA(num_classes=N_CLASSES, num_subjects=DOMAIN_CLASSES).to(DEVICE)
    x = torch.randn(2, N_CHANNELS, int(FS * TMAX), device=DEVICE)

    with torch.no_grad():
        class_logits, domain_logits, z_proj = model(x, lambda_grl=0.0)

    print("Smoke test:")
    print("  input:", tuple(x.shape))
    print("  class logits:", tuple(class_logits.shape))
    print("  domain logits:", tuple(domain_logits.shape))
    print("  projection:", tuple(z_proj.shape))
    print("  params:", sum(p.numel() for p in model.parameters()))

smoke_test()


Smoke test:
  input: (2, 22, 1000)
  class logits: (2, 4)
  domain logits: (2, 109)
  projection: (2, 128)
  params: 51937


## CELL 10 — Training and subject-independent evaluation

`evaluate_large_scale_loso()` is the main experiment.

For each selected held-out subject:
1. choose the test subject;
2. sample up to 99 other available subjects for training;
3. load all selected runs for those subjects;
4. train a fresh model for 100 epochs;
5. ramp the GRL strength from approximately 0 toward 1 using a logistic schedule;
6. optimize classification + domain + contrastive losses;
7. apply AdaBN using all available target trials;
8. evaluate the target subject without labels during adaptation;
9. collect predictions, probabilities, embeddings, and per-epoch losses.

**Important terminology:** the function is named `evaluate_large_scale_loso`, but with 109 subjects it samples 10 test subjects, and each fold uses 99 of the other 108 subjects for training. The remaining 9 subjects are not used in that fold. Therefore this implementation is better described as a **10-fold sampled subject-held-out evaluation** rather than complete 109-fold LOSO. fileciteturn1file0L538-L581

In [14]:
# ============================================================
# CELL 10 — Training and evaluation function
# ============================================================

# -----------------------------
# 5. TRAINING + EVALUATION
# -----------------------------

def evaluate_large_scale_loso(
    data_dir=DATA_DIR,
    total_dataset_subjects=TOTAL_SUBJECTS,
    num_train_subjects=NUM_TRAIN_SUBJECTS,
    num_test_folds=NUM_TEST_FOLDS,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    seed=SEED,
):
    seed_everything(seed)

    available = discover_available_subjects(data_dir, total_dataset_subjects)
    if len(available) < num_test_folds:
        raise RuntimeError(
            f"Only {len(available)} subject folders found in {data_dir}; "
            f"need at least {num_test_folds}."
        )

    print(f"Available subject folders: {len(available)}")

    # Match supplied sampling idea, but restrict to folders that actually exist.
    rng = random.Random(seed)
    test_subjects = rng.sample(available, min(num_test_folds, len(available)))

    fold_rows = []
    pred_rows = []
    epoch_rows = []
    embedding_rows = []

    for fold_idx, test_subject in enumerate(test_subjects, start=1):
        print("=" * 80)
        print(f"FOLD {fold_idx}/{len(test_subjects)} | TEST SUBJECT S{test_subject:03d}")
        print("=" * 80)

        remaining = [s for s in available if s != test_subject]
        train_count = min(num_train_subjects, len(remaining))
        train_subjects = rng.sample(remaining, train_count)

        train_dataset = EEGMMIDB_Dataset(data_dir, subjects=train_subjects)
        test_dataset = EEGMMIDB_Dataset(data_dir, subjects=[test_subject])

        if len(train_dataset) == 0 or len(test_dataset) == 0:
            print("[WARN] Empty fold; skipping.")
            continue

        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=0,
            pin_memory=False
        )

        test_loader = DataLoader(
            test_dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=0,
            pin_memory=False
        )

        model = S3MambaDA(
            num_classes=N_CLASSES,
            num_subjects=total_dataset_subjects
        ).to(DEVICE)

        criterion_cls = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
        criterion_domain = nn.CrossEntropyLoss()
        criterion_supcon = SupConLoss(temperature=SUPCON_TEMP)

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=LR,
            weight_decay=WEIGHT_DECAY
        )

        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=num_epochs,
            eta_min=1e-5
        )

        use_amp = DEVICE.type == "cuda"
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

        for epoch in range(num_epochs):
            model.train()

            epoch_loss = 0.0
            epoch_cls = 0.0
            epoch_domain = 0.0
            epoch_supcon = 0.0
            seen = 0

            total_batches = len(train_loader)

            for batch_idx, (x, y, s) in enumerate(train_loader):
                x = x.to(DEVICE)
                y = y.to(DEVICE)
                s = s.to(DEVICE)

                p = float(batch_idx + epoch * total_batches) / max(
                    1, num_epochs * total_batches
                )
                lambda_grl = 2.0 / (1.0 + np.exp(-10.0 * p)) - 1.0

                optimizer.zero_grad(set_to_none=True)

                with torch.autocast(
                    device_type=DEVICE.type,
                    enabled=use_amp
                ):
                    class_logits, domain_logits, z_proj = model(
                        x, lambda_grl=lambda_grl
                    )

                    loss_cls = criterion_cls(class_logits, y)
                    loss_domain = criterion_domain(domain_logits, s)
                    loss_supcon = criterion_supcon(z_proj, y)

                    loss_total = (
                        loss_cls
                        + DOMAIN_WEIGHT * loss_domain
                        + SUPCON_WEIGHT * loss_supcon
                    )

                scaler.scale(loss_total).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=GRAD_CLIP
                )
                scaler.step(optimizer)
                scaler.update()

                bs = x.size(0)
                seen += bs
                epoch_loss += float(loss_total.detach().cpu()) * bs
                epoch_cls += float(loss_cls.detach().cpu()) * bs
                epoch_domain += float(loss_domain.detach().cpu()) * bs
                epoch_supcon += float(loss_supcon.detach().cpu()) * bs

            scheduler.step()

            epoch_rows.append({
                "fold": fold_idx,
                "test_subject": test_subject,
                "epoch": epoch + 1,
                "loss_total": epoch_loss / max(1, seen),
                "loss_cls": epoch_cls / max(1, seen),
                "loss_domain": epoch_domain / max(1, seen),
                "loss_supcon": epoch_supcon / max(1, seen),
                "lr": optimizer.param_groups[0]["lr"],
            })

            if (epoch + 1) % 10 == 0 or epoch == 0:
                print(
                    f"Epoch {epoch+1:03d}/{num_epochs} | "
                    f"Loss {epoch_loss/max(1,seen):.4f}"
                )

        # Unsupervised target-statistics adaptation.
        model = apply_adabn(
            model,
            test_loader,
            DEVICE,
            adaptation_trials=len(test_dataset)
        )

        model.eval()

        all_preds, all_labels, all_probs, all_embeds, all_subjects = [], [], [], [], []

        with torch.no_grad():
            for x, y, s in test_loader:
                x = x.to(DEVICE)
                y = y.to(DEVICE)

                class_logits, _, z_proj = model(x, lambda_grl=0.0)

                probs = torch.softmax(class_logits, dim=1)
                preds = torch.argmax(probs, dim=1)

                all_preds.extend(preds.cpu().numpy().tolist())
                all_labels.extend(y.cpu().numpy().tolist())
                all_probs.append(probs.cpu().numpy())
                all_embeds.append(z_proj.cpu().numpy())
                all_subjects.extend(s.cpu().numpy().tolist())

        all_probs = np.concatenate(all_probs, axis=0)
        all_embeds = np.concatenate(all_embeds, axis=0)

        acc = accuracy_score(all_labels, all_preds)
        kappa = cohen_kappa_score(all_labels, all_preds)

        report = classification_report(
            all_labels,
            all_preds,
            labels=list(range(N_CLASSES)),
            target_names=CLASS_NAMES,
            output_dict=True,
            zero_division=0
        )

        fold_rows.append({
            "fold": fold_idx,
            "test_subject": test_subject,
            "n_train_trials": len(train_dataset),
            "n_test_trials": len(test_dataset),
            "accuracy": acc,
            "kappa": kappa,
            "precision_macro": report["macro avg"]["precision"],
            "recall_macro": report["macro avg"]["recall"],
            "f1_macro": report["macro avg"]["f1-score"],
        })

        for i in range(len(all_labels)):
            pred_rows.append({
                "fold": fold_idx,
                "test_subject": test_subject,
                "true_label": int(all_labels[i]),
                "pred_label": int(all_preds[i]),
                **{f"prob_{c}": float(all_probs[i, c]) for c in range(N_CLASSES)}
            })

            embedding_rows.append({
                "fold": fold_idx,
                "test_subject": test_subject,
                "true_label": int(all_labels[i]),
                **{
                    f"z_{j}": float(all_embeds[i, j])
                    for j in range(all_embeds.shape[1])
                }
            })

        print(
            f"Subject S{test_subject:03d} | "
            f"Accuracy={acc*100:.2f}% | Kappa={kappa:.4f}"
        )

    # Save all experiment artifacts.
    fold_df = pd.DataFrame(fold_rows)
    pred_df = pd.DataFrame(pred_rows)
    epoch_df = pd.DataFrame(epoch_rows)
    emb_df = pd.DataFrame(embedding_rows)

    fold_df.to_csv(RESULTS_DIR / "fold_metrics.csv", index=False)
    pred_df.to_csv(RESULTS_DIR / "test_predictions.csv", index=False)
    epoch_df.to_csv(RESULTS_DIR / "epoch_history.csv", index=False)
    emb_df.to_csv(RESULTS_DIR / "test_embeddings.csv", index=False)

    summary = {
        "mean_accuracy": float(fold_df["accuracy"].mean()) if len(fold_df) else None,
        "std_accuracy": float(fold_df["accuracy"].std(ddof=0)) if len(fold_df) else None,
        "mean_kappa": float(fold_df["kappa"].mean()) if len(fold_df) else None,
        "std_kappa": float(fold_df["kappa"].std(ddof=0)) if len(fold_df) else None,
        "num_folds_completed": int(len(fold_df)),
        "test_subjects": test_subjects,
        "device": str(DEVICE),
    }

    with open(RESULTS_DIR / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    print("\nFINAL SUMMARY")
    print(json.dumps(summary, indent=2))

    return fold_df, pred_df, epoch_df, emb_df


## CELL 11 — Architecture and training-loss figures

These functions create paper-style figures from the experiment outputs.

`plot_architecture()` draws the conceptual network.
`plot_training_curves()` aggregates losses across folds by epoch and plots total, classification, domain, and SupCon losses. fileciteturn1file0L808-L894

In [15]:
# ============================================================
# CELL 11 — Architecture and training-loss plots
# ============================================================

# ------------------------------------------------------------
# 6. PAPER-READY FIGURES / INFOGRAPHICS
# ------------------------------------------------------------

def plot_architecture():
    fig, ax = plt.subplots(figsize=(15, 7))
    ax.set_xlim(0, 15)
    ax.set_ylim(0, 8)
    ax.axis("off")

    blocks = [
        (0.3, 5.5, 1.7, 1.0, "Raw EEG\n(B,22,1000)"),
        (2.4, 5.5, 1.9, 1.0, "Per-trial\nZ-score"),
        (4.7, 5.5, 2.0, 1.0, "Learnable\nSinc Bank\n10 bands"),
        (7.1, 5.5, 2.0, 1.0, "Dynamic GNN\nAdaptive graph\n22 → 64"),
        (9.5, 5.5, 2.0, 1.0, "BiGRU\nTemporal\nmodeling"),
        (11.9, 5.5, 2.2, 1.0, "SE Attention\n+ mean pool\nz ∈ R64"),
    ]

    for x, y, w, h, txt in blocks:
        r = plt.Rectangle((x, y), w, h, fill=False, linewidth=1.8)
        ax.add_patch(r)
        ax.text(x+w/2, y+h/2, txt, ha="center", va="center", fontsize=11)

    for i in range(len(blocks)-1):
        x1 = blocks[i][0] + blocks[i][2]
        x2 = blocks[i+1][0]
        y = blocks[i][1] + blocks[i][3]/2
        ax.annotate("", xy=(x2, y), xytext=(x1, y),
                    arrowprops=dict(arrowstyle="->", linewidth=1.5))

    # Branches
    ax.plot([13.0, 13.0], [5.5, 3.8], linewidth=1.5)
    ax.annotate("", xy=(10.7, 3.2), xytext=(13.0, 3.8),
                arrowprops=dict(arrowstyle="->", linewidth=1.5))
    ax.annotate("", xy=(13.0, 1.8), xytext=(13.0, 3.8),
                arrowprops=dict(arrowstyle="->", linewidth=1.5))
    ax.annotate("", xy=(6.8, 1.8), xytext=(13.0, 3.8),
                arrowprops=dict(arrowstyle="->", linewidth=1.5))

    outputs = [
        (5.4, 0.7, 2.8, 1.0, "Class Head\n4 MI classes"),
        (9.1, 0.7, 3.2, 1.0, "GRL + Domain Head\nSubject alignment"),
        (3.2, 2.6, 3.6, 1.0, "Projection Head\n128-D SupCon embedding"),
    ]

    for x, y, w, h, txt in outputs:
        r = plt.Rectangle((x, y), w, h, fill=False, linewidth=1.6)
        ax.add_patch(r)
        ax.text(x+w/2, y+h/2, txt, ha="center", va="center", fontsize=10)

    ax.text(7.5, 7.6,
            "S³ Spectral-Spatial Domain-Adaptive EEG Classification Pipeline",
            ha="center", va="center", fontsize=16, fontweight="bold")
    ax.text(7.5, 7.1,
            "Code-aligned architecture: learnable frequency decomposition + dynamic topology + bidirectional GRU + GRL + SupCon + AdaBN",
            ha="center", va="center", fontsize=10)

    fig.tight_layout()
    fig.savefig(FIG_DIR / "Fig1_Architecture.png", dpi=400, bbox_inches="tight")
    plt.close(fig)

def plot_training_curves(epoch_df):
    if epoch_df.empty:
        return

    # Aggregate across folds by epoch.
    g = epoch_df.groupby("epoch").agg({
        "loss_total": "mean",
        "loss_cls": "mean",
        "loss_domain": "mean",
        "loss_supcon": "mean",
    }).reset_index()

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(g["epoch"], g["loss_total"], label="Total loss", linewidth=2)
    ax.plot(g["epoch"], g["loss_cls"], label="Classification loss", linewidth=1.5)
    ax.plot(g["epoch"], g["loss_domain"], label="Domain loss", linewidth=1.5)
    ax.plot(g["epoch"], g["loss_supcon"], label="SupCon loss", linewidth=1.5)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Training Loss Components")
    ax.grid(True, alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIG_DIR / "Fig2_TrainingLoss.png", dpi=400, bbox_inches="tight")
    plt.close(fig)


## CELL 12 — Performance figures

These functions visualize:
- held-out-subject accuracy;
- normalized confusion matrix;
- precision, recall, and F1 for the four classes;
- one-vs-rest ROC curves and AUC.

They read the CSV artifacts generated by the evaluation function. fileciteturn1file0L897-L1007

In [16]:
# ============================================================
# CELL 12 — Accuracy, confusion matrix, class metrics, ROC
# ============================================================

def plot_subject_accuracy(fold_df):
    if fold_df.empty:
        return

    fig, ax = plt.subplots(figsize=(9, 5))
    labels = [f"S{s:03d}" for s in fold_df["test_subject"]]
    ax.bar(labels, fold_df["accuracy"] * 100)
    ax.axhline(25, linestyle="--", linewidth=1.3, label="4-class chance")
    ax.set_ylabel("Accuracy (%)")
    ax.set_xlabel("Held-out subject")
    ax.set_title("Subject-Independent Test Accuracy by Fold")
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "Fig3_FoldAccuracy.png", dpi=400, bbox_inches="tight")
    plt.close(fig)

def plot_confusion(pred_df):
    if pred_df.empty:
        return

    cm = confusion_matrix(
        pred_df["true_label"],
        pred_df["pred_label"],
        labels=list(range(N_CLASSES))
    )

    cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm_norm, interpolation="nearest")
    ax.set_title("Normalized Confusion Matrix")
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_xticks(range(N_CLASSES), CLASS_NAMES, rotation=25, ha="right")
    ax.set_yticks(range(N_CLASSES), CLASS_NAMES)

    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            ax.text(j, i, f"{cm_norm[i, j]*100:.1f}%",
                    ha="center", va="center")

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "Fig4_ConfusionMatrix.png", dpi=400, bbox_inches="tight")
    plt.close(fig)

def plot_class_metrics(pred_df):
    if pred_df.empty:
        return

    report = classification_report(
        pred_df["true_label"],
        pred_df["pred_label"],
        labels=list(range(N_CLASSES)),
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0
    )

    precision = [report[c]["precision"] * 100 for c in CLASS_NAMES]
    recall = [report[c]["recall"] * 100 for c in CLASS_NAMES]
    f1 = [report[c]["f1-score"] * 100 for c in CLASS_NAMES]

    x = np.arange(N_CLASSES)
    width = 0.25

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(x-width, precision, width, label="Precision")
    ax.bar(x, recall, width, label="Recall")
    ax.bar(x+width, f1, width, label="F1")
    ax.set_xticks(x, CLASS_NAMES, rotation=20, ha="right")
    ax.set_ylim(0, 100)
    ax.set_ylabel("Score (%)")
    ax.set_title("Per-Class Classification Metrics")
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "Fig5_ClassMetrics.png", dpi=400, bbox_inches="tight")
    plt.close(fig)

def plot_roc(pred_df):
    if pred_df.empty:
        return

    y_true = pred_df["true_label"].to_numpy()
    y_prob = pred_df[[f"prob_{c}" for c in range(N_CLASSES)]].to_numpy()

    fig, ax = plt.subplots(figsize=(7, 6))

    for c in range(N_CLASSES):
        binary = (y_true == c).astype(int)
        if binary.min() == binary.max():
            continue
        fpr, tpr, _ = roc_curve(binary, y_prob[:, c])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, linewidth=2,
                label=f"{CLASS_NAMES[c]} (AUC={roc_auc:.3f})")

    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("One-vs-Rest ROC Curves")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "Fig6_ROC_AUC.png", dpi=400, bbox_inches="tight")
    plt.close(fig)


## CELL 13 — Embedding visualization and figure generation

`plot_tsne()` reduces the saved 128-D contrastive embeddings to 2-D using t-SNE and colors points by the true EEG class.

`generate_all_figures()` first generates the architecture figure, then checks whether all result CSV files exist. If they do, it generates the remaining six figures and saves them under `results_s3_da/figures/`. fileciteturn2file0L10-L75

In [17]:
# ============================================================
# CELL 13 — t-SNE and figure generation
# ============================================================

def plot_tsne(emb_df, seed=SEED):
    if emb_df.empty:
        return

    z_cols = [c for c in emb_df.columns if c.startswith("z_")]
    if len(emb_df) < 10 or len(z_cols) < 2:
        return

    X = emb_df[z_cols].to_numpy()
    y = emb_df["true_label"].to_numpy()

    perplexity = min(30, max(5, (len(X) - 1) // 3))

    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        init="pca",
        learning_rate="auto",
        random_state=seed
    )

    Z = tsne.fit_transform(X)

    fig, ax = plt.subplots(figsize=(8, 6))
    for c in range(N_CLASSES):
        mask = y == c
        ax.scatter(Z[mask, 0], Z[mask, 1], s=12, label=CLASS_NAMES[c], alpha=0.75)

    ax.set_title("t-SNE Projection of Test-Time Class Embeddings")
    ax.set_xlabel("t-SNE dimension 1")
    ax.set_ylabel("t-SNE dimension 2")
    ax.legend(markerscale=1.5, fontsize=8)
    ax.grid(True, alpha=0.15)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "Fig7_tSNE.png", dpi=400, bbox_inches="tight")
    plt.close(fig)

def generate_all_figures():
    plot_architecture()

    required = [
        "fold_metrics.csv",
        "test_predictions.csv",
        "epoch_history.csv",
        "test_embeddings.csv"
    ]

    if not all((RESULTS_DIR / f).exists() for f in required):
        print("Only architecture figure was generated. Run the experiment first.")
        return

    fold_df = pd.read_csv(RESULTS_DIR / "fold_metrics.csv")
    pred_df = pd.read_csv(RESULTS_DIR / "test_predictions.csv")
    epoch_df = pd.read_csv(RESULTS_DIR / "epoch_history.csv")
    emb_df = pd.read_csv(RESULTS_DIR / "test_embeddings.csv")

    plot_training_curves(epoch_df)
    plot_subject_accuracy(fold_df)
    plot_confusion(pred_df)
    plot_class_metrics(pred_df)
    plot_roc(pred_df)
    plot_tsne(emb_df)

    print("Figures written to:", FIG_DIR.resolve())
    for p in sorted(FIG_DIR.glob("*.png")):
        print(" -", p.name)


## CELL 14 — Run the experiment

Run this cell only after the smoke test succeeds.

For a cheap pipeline check, temporarily set `NUM_EPOCHS` to a small value such as 1–2 in CELL 02. For the full source configuration, restore it to 100.

The original notebook calls figure generation immediately, but the actual training call is left commented out. This cell makes the intended execution order explicit.

In [18]:
# ============================================================
# CELL 14 — Run / generate current figures
# ============================================================

# ------------------------------------------------------------
# 7. RUN
# ------------------------------------------------------------
# For a first smoke test, keep NUM_EPOCHS small (e.g., 2) by changing
# the configuration above, then restore the final value for the paper.
#
# Full experiment:
fold_df, pred_df, epoch_df, emb_df = evaluate_large_scale_loso()
# generate_all_figures()

# Architecture is always safe to generate immediately.
generate_all_figures()


Available subject folders: 109
FOLD 1/1 | TEST SUBJECT S082
[OK] S015 R04 | T1=2 -> class 0 | T2=3 -> class 1 | epochs=15
[OK] S015 R06 | T1=2 -> class 2 | T2=3 -> class 3 | epochs=15
[OK] S015 R08 | T1=2 -> class 0 | T2=3 -> class 1 | epochs=15
[OK] S015 R10 | T1=2 -> class 2 | T2=3 -> class 3 | epochs=15
[OK] S015 R12 | T1=2 -> class 0 | T2=3 -> class 1 | epochs=15
[OK] S015 R14 | T1=2 -> class 2 | T2=3 -> class 3 | epochs=15
[OK] S004 R04 | T1=2 -> class 0 | T2=3 -> class 1 | epochs=15
[OK] S004 R06 | T1=2 -> class 2 | T2=3 -> class 3 | epochs=15
[OK] S004 R08 | T1=2 -> class 0 | T2=3 -> class 1 | epochs=15
[OK] S004 R10 | T1=2 -> class 2 | T2=3 -> class 3 | epochs=15
[OK] S004 R12 | T1=2 -> class 0 | T2=3 -> class 1 | epochs=15
[OK] S004 R14 | T1=2 -> class 2 | T2=3 -> class 3 | epochs=15
[OK] S096 R04 | T1=2 -> class 0 | T2=3 -> class 1 | epochs=15
[OK] S096 R06 | T1=2 -> class 2 | T2=3 -> class 3 | epochs=15
[OK] S096 R08 | T1=2 -> class 0 | T2=3 -> class 1 | epochs=15
[OK] S096 

## CELL 15 — Export paper-ready results text

This helper reads `fold_metrics.csv` and produces a human-readable `paper_results.txt` containing:
- number of completed folds;
- held-out subject IDs;
- mean ± standard deviation of accuracy;
- mean ± standard deviation of Cohen's kappa;
- the full per-fold metrics table.

In [ ]:
# ============================================================
# CELL 15 — Paper results text export
# ============================================================

# ------------------------------------------------------------
# 8. OPTIONAL: PAPER RESULTS TEXT EXPORT
# ------------------------------------------------------------
def export_results_text():
    fold_file = RESULTS_DIR / "fold_metrics.csv"
    if not fold_file.exists():
        print("Run the experiment first.")
        return

    fold_df = pd.read_csv(fold_file)
    if fold_df.empty:
        print("No completed folds.")
        return

    acc_mean = fold_df["accuracy"].mean() * 100
    acc_std = fold_df["accuracy"].std(ddof=0) * 100
    kap_mean = fold_df["kappa"].mean()
    kap_std = fold_df["kappa"].std(ddof=0)

    text = f"""
RESULTS SUMMARY FOR PAPER
=========================
Completed folds: {len(fold_df)}
Subjects: {", ".join("S%03d" % s for s in fold_df["test_subject"])}

Mean Accuracy: {acc_mean:.2f}% ± {acc_std:.2f}%
Mean Cohen's Kappa: {kap_mean:.4f} ± {kap_std:.4f}%

Per-fold:
{fold_df.to_string(index=False)}
""".strip()

    (RESULTS_DIR / "paper_results.txt").write_text(text)
    print(text)

# export_results_text()


## CELL 16 — Execution order and outputs

### Recommended execution order

Run **CELL 01 → CELL 09** first.  
Then run the smoke test.  
For an actual experiment, run **CELL 10**.  
After training completes, run **CELL 13–15**.

### Main artifacts
The experiment writes:
- `fold_metrics.csv` — one row per completed held-out subject;
- `test_predictions.csv` — true/predicted labels and class probabilities;
- `epoch_history.csv` — loss curves and learning rates;
- `test_embeddings.csv` — 128-D contrastive embeddings;
- `summary.json` — aggregate experiment summary;
- `figures/*.png` — architecture, losses, accuracy, confusion matrix, class metrics, ROC, and t-SNE;
- `paper_results.txt` — formatted results text. fileciteturn1file0L778-L805

### Conceptually, the complete learning objective is

`Raw EEG`
→ `Normalization`
→ `Learnable Spectral Decomposition`
→ `Adaptive Spatial Graph`
→ `Bidirectional Temporal Modeling`
→ `Attention`
→ `Shared 64-D Representation`

From the shared representation:

`→ Class head`  for motor-imagery classification  
`→ GRL + domain head`  for subject-invariant learning  
`→ SupCon head`  for discriminative embedding structure

Then:

`Target unlabeled EEG`
→ `AdaBN`
→ `Target prediction`

This is a **domain-adaptive, subject-independent EEG classification pipeline**, with spectral, spatial, temporal, adversarial, contrastive, and test-time-statistics adaptation components.